# The vocabulary, and why it is not the marketing one

This package was ported out of a marketing mix codebase. Everything in it was named for one
industry: channel, spend, geo, KPI. That naming was not a cosmetic problem. It made the
generalizations invisible — a "channel-level saturation curve" is obviously about marketing,
so when the same mathematics was needed for a fertilizer trial, somebody wrote it again.

| marketing (parent) | axiom |
|---|---|
| channel | `Treatment` |
| spend / impressions | `Dose` |
| geo / DMA | `Unit` |
| KPI / sales | `Outcome` |
| control variable | `Covariate` |

Every entity carries a `Dimension` and optionally a unit of measure (a plain string that the
`UnitSystem` knows). `Dose` also carries a `numeraire`. `Population`, `TimeWindow`, and
`Intervention` are the scope-and-intervention vocabulary the estimand facets are built on —
and they each pin down something a variable name would leave to the reader.

In [ ]:
import warnings

import numpy as np

from axiom.core import (
    Covariate,
    D,
    Dose,
    Entity,
    EntityName,
    Intervention,
    LatentSelection,
    Outcome,
    Population,
    TimeWindow,
    Treatment,
    UndimensionedWarning,
    Unit,
    dimension_of,
    dimensionless,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import ORANGE, annotate, caption, compare, lines

enable();  # every axiom result renders itself from here on

In [ ]:
fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD", description="applied nitrogen, costed")
dose = Dose(name="fertilizer_dose", dimension=D.currency, unit="USD", numeraire="USD")
plot = Unit(name="plot", dimension=D.entity, kind="cluster")
yield_total = Outcome(name="yield_total", dimension=D.outcome, unit="kg", aggregation="sum")
rainfall = Covariate(name="rainfall", dimension=dimensionless(), description="standardized")

table(
    [
        [type(e).__name__, e.name, str(e.dim), e.unit]
        for e in (fertilizer, dose, plot, yield_total, rainfall)
    ],
    headers=("entity", "name", "dimension", "unit"),
)

## `Entity` is a protocol, not a base class

The five entity specs are independent classes (composition over inheritance: they share the
`EntityName` field type and one validator function, not a parent). `Entity` is the protocol
any of them satisfies, so a function can accept "any entity" without a hierarchy — and
nobody has to decide whether a `Dose` is-a `Treatment`, which is a question with no true
answer and two plausible ones.

In [ ]:
def describe(e: Entity) -> str:
    return f"{e.name} [{dimension_of(e)}]"

print([describe(e) for e in (fertilizer, plot, rainfall)])
print(isinstance(dose, Entity), Treatment.__mro__[1].__name__)

name: EntityName = "tv-ads.v2"
print(name)

## Undimensioned user entities warn once (D6)

User code may omit the dimension; the entity is then dimensionless and a warning says so.
Anything *shipped* in `src/axiom` must be dimensioned — gate 10 checks that separately, so
the library cannot take the shortcut it offers a user in a hurry.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    loose = Covariate(name="something")
print(loose.dim, "|", [w.category.__name__ for w in caught], "|", issubclass(caught[0].category, UndimensionedWarning))

## Population, window, intervention

Note what each pins down that a variable name would not: strata weights on the population,
the time *basis* on the window, the treatment *version* on the intervention (review B2 —
SUTVA-1).

In [ ]:
north = Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}})
season = TimeWindow(start=0, stop=13, basis="cumulative")
iv = Intervention(doses={"fertilizer": 120.0}, mode="set", version="granular-v2", window=season)

print(north.strata, "|", season.length, season.basis)
print(iv.treatments, iv.mode, iv.version)
print("same dose, different version, different intervention:",
      iv != iv.model_copy(update={"version": "liquid-v1"}))

### A population is a set of weights, and they are the answer

"The effect in the north" is not a number until the mix of strata is stated. Below, the same
two per-stratum effects reweighted for three populations. Nothing about the estimator
changes; the answer moves by more than most experiments could resolve.

In [ ]:
per_stratum = {"clay": 1.6, "loam": 4.2}
populations = {
    "north (30/70 clay/loam)": north.strata["soil"],
    "south (80/20)": {"clay": 0.8, "loam": 0.2},
    "national (50/50)": {"clay": 0.5, "loam": 0.5},
}
averaged = {
    label: sum(per_stratum[s] * w for s, w in weights.items()) for label, weights in populations.items()
}

fig = compare(
    list(averaged), list(averaged.values()),
    highlight="north (30/70 clay/loam)",
    value_fmt="{:.2f}",
    title="The same effect, three populations",
    subtitle="clay plots respond 1.6, loam plots 4.2 — the average is a property of the mix",
    x_title="population-average effect (kg per USD)",
)
caption(fig, "This is why Population carries strata weights rather than a name. Transporting "
             "the north's number to the south is a claim about these bars, and "
             "nbs/identify/03-transport.ipynb is where that claim has to be stated.")

### A window carries its basis for the same reason

`basis="cumulative"` and a per-period reading of the same window are different quantities.
This is the failure from `01-dimensions` in its second most common disguise: not a wrong
unit, but a right unit read over the wrong horizon.

In [ ]:
periods = np.arange(season.start, season.stop)
per_period = np.full(periods.shape, 4.2)
fig = lines(
    periods,
    {"per period": per_period, "cumulative": np.cumsum(per_period)},
    title="One window, two quantities",
    subtitle=f"basis='{season.basis}' over periods {season.start}–{season.stop} — the field names are identical",
    x_title="period", y_title="effect (kg per USD)",
)
annotate(fig, 12, float(np.cumsum(per_period)[-1]), "13× the per-period number")
caption(fig, "A reader handed the second series under the first series' label reports an "
             "effect an order of magnitude too large — and every number in it is real.")

In [ ]:
try:
    Population(name="bad", strata={"soil": {"clay": 0.5, "loam": 0.6}})
except ValueError as e:
    print("refused:", e)
try:
    TimeWindow(start=5, stop=5)
except ValueError as e:
    print("refused:", e)

## A population you can size and cannot list

Every scope object above describes a population the way a survey would: a name, and weights
over covariate levels. That covers the populations you can point at. It does not cover the one
a field experiment usually produces.

The compliers of an encouragement design — the units that took the treatment *because* they
were assigned it — are a real set of people with a real average effect, and their share is
identified from two exposure rates. No covariate distinguishes a member from a never-taker.
You can say how many there are and never which ones, so there is no `strata` entry that
describes them.

`LatentSelection` is that population, and its identity is the response, not the size: a
complier of a letter is not a complier of a phone call, because a different instrument moves a
different set of people.

In [ ]:
letter = LatentSelection(kind="complier", instrument="letter", exposure="attended", share=0.61)
call = LatentSelection(kind="complier", instrument="phone_call", exposure="attended", share=0.40)
never = LatentSelection(kind="never_taker", instrument="letter", exposure="attended")

everybody = Population(name="enrolled")
compliers = Population(name="enrolled", latent=letter)

table([[str(p), p.is_latent, p.content_hash()[:12] + "…"] for p in (everybody, compliers)],
      headers=("population", "latent", "hash"), title="the same name, two populations")
print("compliers of the letter:", letter)
print("same stratum, resized :", letter.same_stratum(letter.model_copy(update={"share": 0.2})))
print("a different instrument:", letter.same_stratum(call), "— different people")
print("a different type      :", letter.same_stratum(never))

`Population` is one of the eight facets of an `Estimand`, so this is not decoration:
`Estimand.transfer_to` reads two latent selections as different populations and refuses to
bridge them, and `meta.commensurable` refuses to *pool* a latent population with the one
around it even where a transfer is licensed. `nbs/meta/02` shows that refusal.

## What this bought you

One vocabulary that a trial, a field experiment, and a media plan can all be written in — so
the mathematics gets written once — and three scope objects that make "for whom, over what
horizon, under which version of the treatment" part of the type rather than part of the
conversation.

`nbs/adapters/` is where a marketing or agronomy vocabulary is mapped onto these entities at
the edge of the system, which is the only place where domain words are allowed to live.